In [3]:
import pandas as pd

In [4]:
data = [[1, 201, '2024-03-01 10:00:00', 'app_open', 'S001', None], [2, 201, '2024-03-01 10:05:00', 'scroll', 'S001', 500], [3, 201, '2024-03-01 10:10:00', 'scroll', 'S001', 750], [4, 201, '2024-03-01 10:15:00', 'scroll', 'S001', 600], [5, 201, '2024-03-01 10:20:00', 'scroll', 'S001', 800], [6, 201, '2024-03-01 10:25:00', 'scroll', 'S001', 550], [7, 201, '2024-03-01 10:30:00', 'scroll', 'S001', 900], [8, 201, '2024-03-01 10:35:00', 'app_close', 'S001', None], [9, 202, '2024-03-01 11:00:00', 'app_open', 'S002', None], [10, 202, '2024-03-01 11:02:00', 'click', 'S002', None], [11, 202, '2024-03-01 11:05:00', 'scroll', 'S002', 400], [12, 202, '2024-03-01 11:08:00', 'click', 'S002', None], [13, 202, '2024-03-01 11:10:00', 'scroll', 'S002', 350], [14, 202, '2024-03-01 11:15:00', 'purchase', 'S002', 50], [15, 202, '2024-03-01 11:20:00', 'app_close', 'S002', None], [16, 203, '2024-03-01 12:00:00', 'app_open', 'S003', None], [17, 203, '2024-03-01 12:10:00', 'scroll', 'S003', 1000], [18, 203, '2024-03-01 12:20:00', 'scroll', 'S003', 1200], [19, 203, '2024-03-01 12:25:00', 'click', 'S003', None], [20, 203, '2024-03-01 12:30:00', 'scroll', 'S003', 800], [21, 203, '2024-03-01 12:40:00', 'scroll', 'S003', 900], [22, 203, '2024-03-01 12:50:00', 'scroll', 'S003', 1100], [23, 203, '2024-03-01 13:00:00', 'app_close', 'S003', None], [24, 204, '2024-03-01 14:00:00', 'app_open', 'S004', None], [25, 204, '2024-03-01 14:05:00', 'scroll', 'S004', 600], [26, 204, '2024-03-01 14:08:00', 'scroll', 'S004', 700], [27, 204, '2024-03-01 14:10:00', 'click', 'S004', None], [28, 204, '2024-03-01 14:12:00', 'app_close', 'S004', None]]
app_events = pd.DataFrame(data, columns={
    "event_id": pd.Series(dtype="int"),
    "user_id": pd.Series(dtype="int"),
    "event_timestamp": pd.Series(dtype="datetime64[ns]"),
    "event_type": pd.Series(dtype="string"),   # varchar -> string
    "session_id": pd.Series(dtype="string"),   # varchar -> string
    "event_value": pd.Series(dtype="int")
})

In [11]:
app_events['if_scroll'] = (app_events['event_type'] == 'scroll').astype(int)
app_events['if_click'] = (app_events['event_type'] == 'click').astype(int)
app_events['if_purchase'] = (app_events['event_type'] == 'purchase').astype(int)

In [13]:
app_events.head(10)

,event_id,user_id,event_timestamp,event_type,session_id,event_value,if_scroll,if_click,if_purchase
0,1,201,2024-03-01 10:00:00,app_open,S001,NaN,0,0,0
1,2,201,2024-03-01 10:05:00,scroll,S001,500.0,1,0,0
2,3,201,2024-03-01 10:10:00,scroll,S001,750.0,1,0,0
3,4,201,2024-03-01 10:15:00,scroll,S001,600.0,1,0,0
4,5,201,2024-03-01 10:20:00,scroll,S001,800.0,1,0,0
5,6,201,2024-03-01 10:25:00,scroll,S001,550.0,1,0,0
6,7,201,2024-03-01 10:30:00,scroll,S001,900.0,1,0,0
7,8,201,2024-03-01 10:35:00,app_close,S001,NaN,0,0,0
8,9,202,2024-03-01 11:00:00,app_open,S002,NaN,0,0,0
9,10,202,2024-03-01 11:02:00,click,S002,NaN,0,1,0


In [17]:
res = app_events.groupby(['session_id', 'user_id']).agg(
    first_time = ('event_timestamp', 'first'),
    last_time = ('event_timestamp', 'last'),
    scroll_count = ('if_scroll', 'sum'),
    click_count=('if_click', 'sum'),
    purchase_count=('if_purchase', 'sum')
).reset_index()

In [26]:
res['session_duration_minutes'] = (pd.to_datetime(res['last_time']) - pd.to_datetime(res['first_time'])).dt.total_seconds() / 60

In [28]:
res['click_count_ratio'] = res['click_count'] / res['scroll_count']

In [32]:
res = res[(res['session_duration_minutes'] > 30) & (res['click_count_ratio'] < 0.2) & (res['purchase_count'] == 0)]
res[['session_id', 'user_id', 'session_duration_minutes', 'scroll_count']].sort_values(['scroll_count', 'session_id'], ascending=[0,1])

,session_id,user_id,session_duration_minutes,scroll_count
0,S001,201,35.0,6


In [22]:
pd.to_datetime(res['last_time'])

0   2024-03-01 10:35:00
1   2024-03-01 11:20:00
2   2024-03-01 13:00:00
3   2024-03-01 14:12:00
Name: last_time, dtype: datetime64[ns]

In [33]:
data = [[1, 101, 'Python Basics', '2024-01-05', 5], [1, 102, 'SQL Fundamentals', '2024-02-10', 4], [1, 103, 'JavaScript', '2024-03-15', 5], [1, 104, 'React Basics', '2024-04-20', 4], [1, 105, 'Node.js', '2024-05-25', 5], [1, 106, 'Docker', '2024-06-30', 4], [2, 101, 'Python Basics', '2024-01-08', 4], [2, 104, 'React Basics', '2024-02-14', 5], [2, 105, 'Node.js', '2024-03-20', 4], [2, 106, 'Docker', '2024-04-25', 5], [2, 107, 'AWS Fundamentals', '2024-05-30', 4], [3, 101, 'Python Basics', '2024-01-10', 3], [3, 102, 'SQL Fundamentals', '2024-02-12', 3], [3, 103, 'JavaScript', '2024-03-18', 3], [3, 104, 'React Basics', '2024-04-22', 2], [3, 105, 'Node.js', '2024-05-28', 3], [4, 101, 'Python Basics', '2024-01-12', 5], [4, 108, 'Data Science', '2024-02-16', 5], [4, 109, 'Machine Learning', '2024-03-22', 5]]
course_completions = pd.DataFrame(data, columns={
    "user_id": pd.Series(dtype="int"),
    "course_id": pd.Series(dtype="int"),
    "course_name": pd.Series(dtype="string"),           # corresponds to SQL VARCHAR
    "completion_date": pd.Series(dtype="datetime64[ns]"),  # corresponds to SQL DATE
    "course_rating": pd.Series(dtype="Int64")           # corresponds to SQL INT (nullable)
})

In [59]:
top_students = course_completions.groupby('user_id').agg(
    course_count=('course_id', 'count'),
    avg_rating=('course_rating', 'mean')
).reset_index()

In [61]:
top_students = top_students[(top_students['avg_rating']>= 4) & (top_students['course_count'] >=5)]
top_students = top_students['user_id'].to_list()
filtered_courses = course_completions[course_completions['user_id'].isin(top_students)].sort_values(['user_id','completion_date'])

In [62]:
filtered_courses

,user_id,course_id,course_name,completion_date,course_rating
0,1,101,Python Basics,2024-01-05,5
1,1,102,SQL Fundamentals,2024-02-10,4
2,1,103,JavaScript,2024-03-15,5
3,1,104,React Basics,2024-04-20,4
4,1,105,Node.js,2024-05-25,5
5,1,106,Docker,2024-06-30,4
6,2,101,Python Basics,2024-01-08,4
7,2,104,React Basics,2024-02-14,5
8,2,105,Node.js,2024-03-20,4
9,2,106,Docker,2024-04-25,5


In [63]:
from collections import defaultdict
course_freq = defaultdict(int)
for (user_id), g in filtered_courses.groupby('user_id'):
    extract_freq(g['course_name'])
    # break


In [64]:
def extract_freq(s_course):
    courses = s_course.to_list()
    for i in range(len(courses)-1):
        course_freq[(courses[i], courses[i+1])] += 1

In [65]:
res = []
for course_pair, count in course_freq.items():
    res.append({'first_course': course_pair[0], 'second_course': course_pair[1], 'transition_count': count})

In [68]:
result = pd.DataFrame(res).sort_values(['transition_count', 'first_course', 'second_course'], ascending=[0, 1, 1])

In [71]:
result['1st_lower'] = result['first_course'].map(lambda x: x.lower())
result['2nd_lower'] = result['second_course'].map(lambda x: x.lower())

In [73]:
result.sort_values(['transition_count', '1st_lower', '2nd_lower'], ascending=[0, 1, 1])[['first_course', 'second_course', 'transition_count']]

,first_course,second_course,transition_count
4,Node.js,Docker,2
3,React Basics,Node.js,2
6,Docker,AWS Fundamentals,1
2,JavaScript,React Basics,1
5,Python Basics,React Basics,1
0,Python Basics,SQL Fundamentals,1
1,SQL Fundamentals,JavaScript,1


In [108]:
data = [[1, '2024-01-01', 'login'], [1, '2024-01-02', 'login'], [1, '2024-01-03', 'login'], [1, '2024-01-04', 'login'], [1, '2024-01-05', 'login'], [1, '2024-01-06', 'logout'], [2, '2024-01-01', 'click'], [2, '2024-01-02', 'click'], [2, '2024-01-03', 'click'], [2, '2024-01-04', 'click'], [3, '2024-01-01', 'view'], [3, '2024-01-02', 'view'], [3, '2024-01-03', 'view'], [3, '2024-01-04', 'view'], [3, '2024-01-05', 'view'], [3, '2024-01-06', 'view'], [3, '2024-01-07', 'view']]
activity = pd.DataFrame(data, columns={
    "user_id": pd.Series(dtype="int"),
    "action_date": pd.Series(dtype="datetime64[ns]"),
    "action": pd.Series(dtype="string")
})

In [ ]:
activity['min_date'] = activity.action_date.min()
activity['date_diff'] = (pd.to_datetime(activity['action_date']) - pd.to_datetime(activity['min_date'])).dt.days
activity['day_count'] = activity.groupby(['user_id', 'action_date'])['action'].transform('count')
activity = activity[activity['day_count'] == 1]
activity['streak'] = activity.date_diff - activity.index

In [114]:
activity

,user_id,action_date,action,min_date,date_diff,day_count,streak
0,1,2024-01-01,login,2024-01-01,0,1,0
1,1,2024-01-02,login,2024-01-01,1,1,0
2,1,2024-01-03,login,2024-01-01,2,1,0
3,1,2024-01-04,login,2024-01-01,3,1,0
4,1,2024-01-05,login,2024-01-01,4,1,0
5,1,2024-01-06,logout,2024-01-01,5,1,0
6,2,2024-01-01,click,2024-01-01,0,1,-6
7,2,2024-01-02,click,2024-01-01,1,1,-6
8,2,2024-01-03,click,2024-01-01,2,1,-6
9,2,2024-01-04,click,2024-01-01,3,1,-6


In [122]:
res = activity.groupby(['user_id', 'action', 'streak']).agg(
    streak_length=('action_date', 'count'),
    start_date=('action_date', 'min'),
    end_date =('action_date', 'max')
).reset_index()

In [124]:
res[res['streak_length'] >=5].sort_values(['streak_length', 'user_id'], ascending=[0, 1]).drop_duplicates('user_id').drop('streak', axis=1)

,user_id,action,streak_length,start_date,end_date
3,3,view,7,2024-01-01,2024-01-07
0,1,login,5,2024-01-01,2024-01-05
